# QELM on the measurement record M: TFIM, Haar and Clifford dynamics

Issue #171. The experiment of `03_QELM_M_alpha_ratio.ipynb` (`experiments/QELM_M/exp_QELM_M_ratio.py`), run with three unitary dynamics on the system qubits instead of one:

- `TFIM`: the disordered transverse-field Ising model, a new disorder per sample, the baseline;
- `haar_circuit`: a brickwork circuit of Haar-random two-qubit gates, drawn from U(4) (`QELM.models.circuits.HaarBrickworkCircuit`), no conserved quantity;
- `clifford_circuit`: the same brickwork with two-qubit Clifford gates, drawn uniformly from the 11 520 elements of the group (`QELM.models.circuits.CliffordBrickworkCircuit`).

Everything else is the TFIM run: the ancilla `R` entangled with the charge of the system, the system qubits measured in Z with probability `p` after each of the `nbr_time_evolution` steps, the record `M` as features (0 for an unmeasured qubit), the measured label of `R` (the initial charge sector, ±1) as target, a ridge readout with `n_params = nbr_time_evolution * Ns + 1` weights and `n_train = alpha_ratio * n_params` training samples. For a circuit, one time-evolution step is `layers_per_step` brickwork layers (1 by default, each a full period: gates on the even bonds, then on the odd ones), and every sample gets new gates, as every TFIM sample gets a new disorder (`fixed_instance: false`, see `conf/models/haar_circuit.yaml`).

The sweep is the grid of notebook 03, `p` and `alpha_ratio`, once per model. The figures:

1. the metrics against the ratio, one row of panels per model and `N`, to locate the double-descent peak at `alpha_ratio = 1`;
2. the heatmaps over `p` and the ratio, one row per model and `N` (issue #164);
3. the comparison of the three models on the same axes: the balanced accuracy of the readout and the accuracy of the Bayes classifier `sign(<Z_0>)` against `p` at the largest ratio, and the NMSE against the ratio at `p = 1`. The Bayes accuracy (`accuracy_optimal`) is what the record holds about the label, the purification side of the measurement-induced transition; the gap between it and the readout is what a linear readout of `M` cannot extract.

In [ ]:
import os
from pathlib import Path

# Jupyter starts in the notebook's folder. Move to the repository root (the
# folder that contains pyproject.toml) so conf/, experiments/ and outputs/
# resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here, or through the environment variable of the same name (this is what the test `tests/test_qelm_m_circuits_notebook.py` does for a small sweep). `MODELS` lists the dynamics to compare, the `models` values of `conf/models/`. The other parameters are those of notebook 03. Each sweep goes to this notebook's own folder, `outputs/QELM_M_circuits/<date_time>/`, with one subfolder per model; `RUN = False` reads back the latest sweep of this folder, and `RESUME` set to the path of an earlier sweep folder completes it (the runs already done are skipped, issue #168).

The default grid has `11 x 12` points per model, seed and `N`, so `3 x 3 x 132 = 1188` runs at `N = 4`, a few seconds each.

In [ ]:
# Unitary dynamics to compare, models values of conf/models/.
MODELS = os.environ.get("MODELS", "TFIM haar_circuit clifford_circuit")
N_VALUES = os.environ.get("N_VALUES", "4")  # total qubits N, ancilla included
SEEDS = os.environ.get("SEEDS", "42 43 44")
# Measurement probability, 0 to 1 (the y axis of the heatmaps).
P_VALUES = os.environ.get("P_VALUES", "0.0 0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1.0")
# The values of p drawn as curves against the ratio.
CURVE_P_VALUES = os.environ.get("CURVE_P_VALUES", "0.2 0.5 1.0")
# Training samples per weight, n_train / n_params (experiment.alpha_ratio),
# 0.25 to 10 on a log scale.
ALPHA_RATIOS = os.environ.get(
    "ALPHA_RATIOS", "0.25 0.35 0.5 0.71 1.0 1.41 2.0 2.83 4.0 5.66 8.0 10.0"
)
N_TEST = os.environ.get("N_TEST", "200")  # test samples per run
ALPHA = os.environ.get("ALPHA", "1e-3")  # ridge penalty (training.alpha)
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"
# Sweep folder to complete (runs already done are skipped); "" for a new sweep.
RESUME = os.environ.get("RESUME", "")

## Launch the sweeps

One sweep of `experiments/QELM_M/exp_QELM_M_ratio.sh` per model, with `MODEL` set, each in its own subfolder of the sweep folder.

In [ ]:
from QELM.utils import latest_sweep, new_sweep_dir, run_with_progress

# This notebook's own output folder (issue #151).
EXPERIMENT = "QELM_M_circuits"
models = MODELS.split()

if RUN:
    sweep_dir = Path(RESUME) if RESUME else new_sweep_dir(OUT_ROOT, EXPERIMENT)
    n_runs = (
        len(N_VALUES.split())
        * len(SEEDS.split())
        * len(P_VALUES.split())
        * len(ALPHA_RATIOS.split())
    )
    for model in models:
        print(f"model: {model}")
        env = {
            **os.environ,
            "MODEL": model,
            "N_VALUES": N_VALUES,
            "SEEDS": SEEDS,
            "P_VALUES": P_VALUES,
            "ALPHA_RATIOS": ALPHA_RATIOS,
            "N_TEST": N_TEST,
            "ALPHA": ALPHA,
            "OUT_DIR": str(sweep_dir / model),
        }
        # One bar per model, one step per run.
        run_with_progress(
            ["bash", "experiments/QELM_M/exp_QELM_M_ratio.sh"], total=n_runs, env=env
        )
        failed = sweep_dir / model / "failed_runs.txt"
        if failed.is_file():
            print(
                f"WARNING: failed runs, missing from the figures:\n{failed.read_text()}"
                f'Set RESUME = "{sweep_dir}" and run again to retry them.'
            )
else:
    # RUN = False: the latest sweep of this notebook only.
    sweep_dir = latest_sweep(OUT_ROOT / EXPERIMENT)
print(f"sweep: {sweep_dir}")

## Read back the metrics

One table per model, the mean over the seeds at each `(N, p, ratio)`, with the ratio actually used, `n_train / n_features`, from each run's `metrics.json`. The model of each run is read back from its config and checked against its folder.

In [ ]:
from QELM.utils import load_sweep, summarize

# (key in metrics.json, label, log scale).
METRICS = [
    ("nmse", "NMSE (test)", True),
    ("r2", "R2 (test)", False),
    ("balanced_accuracy", "balanced accuracy (test)", False),
]
SUMMARY = [k for k, _, _ in METRICS] + ["accuracy", "accuracy_optimal", "S_vN_mean"]
tables = {}
for model in models:
    rows = load_sweep(
        sweep_dir / model,
        keys=(
            ("N", "N"),
            ("seed", "seed"),
            ("p", "projective_measurement.p"),
            ("model", "models"),
        ),
    )
    assert {r["model"] for r in rows} == {model}, sweep_dir / model
    for r in rows:
        r["ratio"] = r["n_train"] / r["n_features"]
    tables[model] = summarize(rows, by=("N", "p", "ratio"), metrics=SUMMARY)
    print(f"{model}: {len(rows)} runs, {len(tables[model])} grid points")

## Metrics against the ratio

One row of panels per model and `N`, one panel per metric, one curve per `p` of `CURVE_P_VALUES`, the mean over the seeds with the standard error. The dashed line marks the interpolation threshold `alpha_ratio = 1`; on the balanced accuracy, the dotted gray line marks chance, 0.5. Saved as `metrics_vs_alpha_ratio.png` and `.json` in the sweep folder.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter

from QELM.utils.plots import (
    color_map,
    draw_curve,
    save_figure,
    sizes,
    subplots,
    sweep_curve,
)

Ns = sizes(tables[models[0]])
panel_rows = [(model, N) for model in models for N in Ns]


def ratio_ticks(ax, ratios):
    """Label the log ratio axis at the powers of 2 within the swept ratios."""
    ticks = [2.0**k for k in range(-6, 7) if min(ratios) <= 2.0**k <= max(ratios)]
    ax.set_xticks(ticks, labels=[f"{t:g}" for t in ticks])
    ax.xaxis.set_minor_formatter(NullFormatter())


fig, axes = subplots(
    len(panel_rows),
    len(METRICS),
    figsize=(6.5 * len(METRICS), 5.0 * len(panel_rows)),
    squeeze=False,
)
for row, (model, N) in zip(axes, panel_rows, strict=True):
    table = tables[model]
    ps = sorted({r["p"] for r in table if r["p"] in map(float, CURVE_P_VALUES.split())})
    if not ps:  # none of CURVE_P_VALUES was swept: draw every p
        ps = sorted({r["p"] for r in table})
    colors = color_map([f"p = {p:g}" for p in ps])
    for p in ps:
        rows_p = [r for r in table if r["p"] == p]
        label = f"p = {p:g}"
        for (key, _, log), ax in zip(METRICS, row, strict=True):
            curve = sweep_curve(rows_p, N, "ratio", key)
            draw_curve(ax, curve, colors[label], label=label, log=log)
    for (key, label, _), ax in zip(METRICS, row, strict=True):
        ax.axvline(1.0, color="black", linestyle="--", label="interpolation threshold")
        if key == "balanced_accuracy":
            ax.axhline(0.5, color="gray", linestyle=":", label="chance")
        ax.set_xscale("log")
        ratio_ticks(ax, [r["ratio"] for r in table if r["N"] == N])
        ax.set(
            xlabel="alpha_ratio = n_train / n_params",
            ylabel=label,
            title=f"{label}, {model}, N = {N}",
        )
        ax.legend()
fig.suptitle("QELM on M: metrics against the sample-to-parameter ratio, per dynamics")
figure_path = save_figure(
    fig, sweep_dir, "metrics_vs_alpha_ratio", data={"tables": tables}
)
print(f"figure: {figure_path}")
plt.show()

## Heatmaps over p and the ratio

The heatmaps of notebook 03 (issues #164 and #167), one row per model and `N`: the mean over the seeds at each grid point, `alpha_ratio` on a logarithmic x axis and `p` on the y axis, the dashed white line at `alpha_ratio = 1`. Color scales: NMSE logarithmic, R2 from -1 to 1, accuracies from 0.5 to 1. Saved as `heatmaps_p_alpha_ratio.png` and `.json`.

In [ ]:
from QELM.utils.plots import draw_heatmap, sweep_grid

# (key in metrics.json, label, log color scale, bottom and top of the scale).
HEATMAPS = [
    ("nmse", "NMSE (test)", True, None, None),
    ("r2", "R2 (test)", False, -1.0, 1.0),
    ("accuracy", "accuracy (test)", False, 0.5, 1.0),
    ("balanced_accuracy", "balanced accuracy (test)", False, 0.5, 1.0),
]
fig_maps, axes_maps = subplots(
    len(panel_rows),
    len(HEATMAPS),
    figsize=(5.5 * len(HEATMAPS), 4.6 * len(panel_rows)),
    squeeze=False,
)
grids = {}
for row, (model, N) in zip(axes_maps, panel_rows, strict=True):
    for (key, label, log_color, vmin, vmax), ax in zip(HEATMAPS, row, strict=True):
        grid = sweep_grid(tables[model], "ratio", "p", key, size=N)
        grids[(model, N, key)] = grid
        draw_heatmap(
            ax, grid, label=label, log_x=True, log_color=log_color, vmin=vmin, vmax=vmax
        )
        ratio_ticks(ax, grid.x)
        ax.axvline(1.0, color="white", linestyle="--", label="interpolation threshold")
        ax.set(
            xlabel="alpha_ratio = n_train / n_params",
            ylabel="measurement probability p",
            title=f"{label}, {model}, N = {N}",
        )
fig_maps.suptitle("QELM on M: metrics over p and the sample-to-parameter ratio")
heatmap_path = save_figure(
    fig_maps, sweep_dir, "heatmaps_p_alpha_ratio", data={"tables": tables}
)
print(f"figure: {heatmap_path}")
plt.show()

## The three dynamics compared

One row of panels per `N`, one curve per model:

- balanced accuracy of the readout against `p`, at the largest ratio of the sweep (the end of the double descent, where finite sampling matters least);
- accuracy of the Bayes classifier `sign(<Z_0>)` against `p` (`accuracy_optimal`), which knows the unitary of each sample: what the record holds about the label. It does not depend on the ratio, so it is averaged over the ratios as well as the seeds;
- NMSE against the ratio at the largest `p` of the sweep, to compare the double-descent peaks.

The gap between the Bayes accuracy and the readout is the information that the record holds and a linear readout of `M`, the same for every sample, does not extract. Saved as `comparison_models.png` and `.json`.

In [ ]:
model_colors = color_map(models)
COMPARISON = [
    ("balanced_accuracy", "balanced accuracy (test)"),
    ("accuracy_optimal", "Bayes accuracy, sign <Z_0>"),
    ("nmse", "NMSE (test)"),
]
fig_cmp, axes_cmp = subplots(
    len(Ns),
    len(COMPARISON),
    figsize=(6.5 * len(COMPARISON), 5.0 * len(Ns)),
    squeeze=False,
)
comparison = {}
for row, N in zip(axes_cmp, Ns, strict=True):
    ax_bal, ax_opt, ax_nmse = row
    for model in models:
        table = [r for r in tables[model] if r["N"] == N]
        ratio_max = max(r["ratio"] for r in table)
        p_max = max(r["p"] for r in table)
        # Balanced accuracy against p at the largest ratio.
        at_ratio = [r for r in table if r["ratio"] == ratio_max]
        curve = sweep_curve(at_ratio, N, "p", "balanced_accuracy")
        draw_curve(ax_bal, curve, model_colors[model], label=model)
        # Bayes accuracy against p, averaged over every ratio and seed.
        by_p = summarize(
            [{**r, "accuracy_optimal": r["accuracy_optimal_mean"]} for r in table],
            by=("N", "p"),
            metrics=["accuracy_optimal"],
        )
        curve_opt = sweep_curve(by_p, N, "p", "accuracy_optimal")
        draw_curve(ax_opt, curve_opt, model_colors[model], label=model)
        # NMSE against the ratio at the largest p.
        at_p = [r for r in table if r["p"] == p_max]
        curve_nmse = sweep_curve(at_p, N, "ratio", "nmse")
        draw_curve(ax_nmse, curve_nmse, model_colors[model], label=model, log=True)
        comparison[(model, N)] = {
            "ratio_max": ratio_max,
            "p_max": p_max,
            "balanced_accuracy_vs_p": [
                (r["p"], r["balanced_accuracy_mean"]) for r in curve.rows
            ],
            "accuracy_optimal_vs_p": [
                (r["p"], r["accuracy_optimal_mean"]) for r in curve_opt.rows
            ],
            "nmse_vs_ratio": [(r["ratio"], r["nmse_mean"]) for r in curve_nmse.rows],
        }
    for ax in (ax_bal, ax_opt):
        ax.axhline(0.5, color="gray", linestyle=":", label="chance")
        ax.set(xlabel="measurement probability p", ylim=(0.4, 1.02))
    ax_bal.set(
        ylabel="balanced accuracy (test)",
        title=f"readout at alpha_ratio = {ratio_max:.3g}, N = {N}",
    )
    ax_opt.set(ylabel="Bayes accuracy", title=f"what the record holds, N = {N}")
    ax_nmse.axvline(1.0, color="black", linestyle="--", label="interpolation threshold")
    ax_nmse.set_xscale("log")
    ratio_ticks(ax_nmse, [r["ratio"] for r in tables[models[0]] if r["N"] == N])
    ax_nmse.set(
        xlabel="alpha_ratio = n_train / n_params",
        ylabel="NMSE (test)",
        title=f"NMSE at p = {p_max:g}, N = {N}",
    )
    for ax in row:
        ax.legend()
fig_cmp.suptitle("QELM on M: TFIM, Haar and Clifford dynamics")
comparison_path = save_figure(
    fig_cmp,
    sweep_dir,
    "comparison_models",
    data={"comparison": {f"{m} N={N}": v for (m, N), v in comparison.items()}},
)
print(f"figure: {comparison_path}")
plt.show()